# Inclinação Política com BERTimbau — Frente 2 do Projeto

**Atividade Prática — Deep Learning para Demandas Reais**

Este notebook fecha a **segunda frente** do projeto: classificar a **inclinação ideológica**
(esquerda / direita) em **discursos parlamentares**, usando o **BERTimbau** (BERT pré-treinado em
texto formal em português). Tudo é feito aqui no Colab:

1. **coleta** os discursos direto da API de dados abertos da Câmara dos Deputados;
2. rotula cada discurso pela **ideologia do partido** do deputado (supervisão distante);
3. faz **split agrupado por deputado** (sem vazamento de identidade do autor);
4. faz **fine-tuning do BERTimbau** e avalia com macro-F1, matriz de confusão e ROC-AUC.

> **Antes de rodar:** `Ambiente de execução → Alterar tipo → GPU`, depois `Executar tudo`.
> A coleta leva alguns minutos; o treino, mais alguns numa GPU T4.


## 1. Dependências e GPU

In [ ]:
!pip install -q -U transformers accelerate scikit-learn
import torch
print("GPU:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Coletar discursos da Câmara dos Deputados (API pública)

Coletamos discursos da legislatura atual dentro de uma janela de datas. Para manter o tempo
razoável, há limites por deputado e um alvo total. Aumente `MAX_POR_DEP`/`ALVO` para mais dados.

In [ ]:
import requests, time, random
API = "https://dadosabertos.camara.leg.br/api/v2"
H = {"Accept": "application/json"}
LEGISLATURA = 57            # 57 = 2023-2027
DATA_INI, DATA_FIM = "2023-02-01", "2024-12-31"
MIN_PALAVRAS = 40          # descarta discursos curtos/protocolares
MAX_POR_DEP = 12           # discursos por deputado (limita tempo)
ALVO = 3000                # alvo total de discursos

# Mapeamento partido -> ideologia (eixo binário). Fonte: Bolognesi et al. (2023).
MAPA = {
 "PT":"esquerda","PSOL":"esquerda","PCDOB":"esquerda","PSB":"esquerda","PDT":"esquerda","REDE":"esquerda",
 "PL":"direita","PP":"direita","REPUBLICANOS":"direita","UNIÃO":"direita","NOVO":"direita",
 "PSC":"direita","PSDB":"direita","PSL":"direita","PRTB":"direita",
}

def get(url):
    for i in range(4):
        try:
            return requests.get(url, headers=H, timeout=30).json()
        except Exception:
            time.sleep(1.5*(i+1))
    return {"dados":[], "links":[]}

# lista de deputados
deps=[]; url=f"{API}/deputados?idLegislatura={LEGISLATURA}&ordem=ASC&ordenarPor=nome&itens=100"
while url:
    d=get(url); deps+=d.get("dados",[])
    url=next((l["href"] for l in d.get("links",[]) if l["rel"]=="next"), None)
print("deputados:", len(deps))

random.seed(42); random.shuffle(deps)
linhas=[]
for dep in deps:
    part=(dep.get("siglaPartido") or "").strip().upper()
    ideo=MAPA.get(part)
    if not ideo: continue
    n=0
    u=f"{API}/deputados/{dep['id']}/discursos?itens=50&ordenarPor=dataHoraInicio&ordem=DESC&dataInicio={DATA_INI}&dataFim={DATA_FIM}"
    while u and n<MAX_POR_DEP:
        d=get(u)
        for disc in d.get("dados",[]):
            txt=(disc.get("transcricao") or "").strip()
            if len(txt.split())>=MIN_PALAVRAS:
                linhas.append({"texto":txt,"partido":part,"ideologia":ideo,"id_deputado":dep["id"]})
                n+=1
                if n>=MAX_POR_DEP: break
        u=next((l["href"] for l in d.get("links",[]) if l["rel"]=="next"), None)
    if len(linhas)>=ALVO: break

import pandas as pd
dpol=pd.DataFrame(linhas)
print("discursos coletados:", len(dpol))
print(dpol["ideologia"].value_counts())
dpol.to_csv("discursos_camara.csv", index=False)
dpol.head(2)

## 2b. Limpeza CRÍTICA — remover cabeçalho e siglas (evita vazamento de rótulo)

A transcrição oficial começa com um cabeçalho que contém a **sigla do partido** do orador, no formato (Bloco/PDT - MA. Sem revisão do orador.). Se não removermos, o modelo apenas lê o partido e o resultado fica irrealista (~0,99 = vazamento de rótulo). A célula abaixo remove o cabeçalho e as siglas partidárias, deixando só o conteúdo do discurso.

In [ ]:
import re
SIG = r'\b(PT|PL|PP|PSOL|PCdoB|PCDOB|PSB|PDT|REDE|REPUBLICANOS|UNIÃO|UNIAO|NOVO|PSC|PSDB|PSL|PRTB|MDB|PSD)\b'
def limpar(t):
    t = re.sub(r'^.*?\)\s*-\s*', '', str(t), count=1)   # remove o cabecalho '(Partido-UF...) - '
    t = re.sub(SIG, '', t, flags=re.I)                 # remove siglas partidarias do corpo
    return t
dpol['texto'] = dpol['texto'].map(limpar)
print('Texto limpo. Amostra:')
print(dpol['texto'].iloc[0][:200])

## 3. Rótulo, balanceamento e split AGRUPADO POR DEPUTADO

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
dpol["label"]=(dpol["ideologia"]=="direita").astype(int)
# balancear por downsampling da classe majoritaria
n_min=dpol["label"].value_counts().min()
dpol=pd.concat([g.sample(n_min, random_state=42) for _,g in dpol.groupby("label")]).reset_index(drop=True)
print("apos balancear:", dpol["label"].value_counts().to_dict())
# split por deputado (treino/teste disjuntos por autor) - evita vazamento de estilo
gss=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_i, te_i = next(gss.split(dpol, dpol["label"], groups=dpol["id_deputado"]))
tr, te = dpol.iloc[tr_i], dpol.iloc[te_i]
# subconjunto de validacao a partir do treino (tambem por deputado)
gss2=GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr2_i, va_i = next(gss2.split(tr, tr["label"], groups=tr["id_deputado"]))
va = tr.iloc[va_i]; tr = tr.iloc[tr2_i]
print("treino/val/teste:", len(tr), len(va), len(te))

## 4. Tokenização (BERTimbau — texto formal em português)

In [ ]:
from transformers import AutoTokenizer
MODELO = "neuralmind/bert-base-portuguese-cased"
MAX_LEN = 256   # discursos longos sao truncados; janelamento fica como melhoria futura
tok = AutoTokenizer.from_pretrained(MODELO)

class DS(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.t=frame["texto"].astype(str).tolist(); self.y=frame["label"].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        e=tok(self.t[i], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        it={k:v.squeeze(0) for k,v in e.items()}; it["labels"]=torch.tensor(self.y[i]); return it
ds_tr, ds_va, ds_te = DS(tr), DS(va), DS(te)

## 5. Modelo, perda ponderada e métricas

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, roc_auc_score
ytr=tr["label"].values; n=len(ytr); n1=int(ytr.sum()); n0=n-n1
pesos=torch.tensor([n/(2*max(n0,1)), n/(2*max(n1,1))], dtype=torch.float)

def compute_metrics(p):
    logits,labels=p; preds=np.argmax(logits,axis=-1)
    prob=torch.softmax(torch.tensor(logits),dim=-1)[:,1].numpy()
    return {"macro_f1":f1_score(labels,preds,average="macro",zero_division=0),
            "accuracy":accuracy_score(labels,preds),
            "roc_auc":roc_auc_score(labels,prob)}

class TrainerP(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels=inputs.pop("labels"); out=model(**inputs)
        loss=torch.nn.CrossEntropyLoss(weight=pesos.to(out.logits.device))(out.logits, labels)
        return (loss,out) if return_outputs else loss
model=AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)

## 6. Fine-tuning

In [ ]:
common=dict(output_dir="bertimbau_pol", learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01, warmup_ratio=0.1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    greater_is_better=True, logging_steps=25, report_to="none",
    fp16=torch.cuda.is_available(), seed=42)
try:
    args=TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common)
except TypeError:
    args=TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common)
trainer=TrainerP(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va, compute_metrics=compute_metrics)
trainer.train()

## 7. Avaliação no teste

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
pred=trainer.predict(ds_te); logits=pred.predictions
yhat=np.argmax(logits,axis=-1); prob=torch.softmax(torch.tensor(logits),dim=-1)[:,1].numpy()
yte=te["label"].values
print(classification_report(yte, yhat, target_names=["esquerda","direita"], digits=4))
macro=f1_score(yte,yhat,average="macro"); auc=roc_auc_score(yte,prob); cm=confusion_matrix(yte,yhat)
print(f"macro-F1 = {macro:.4f} | ROC-AUC = {auc:.4f}")
json.dump({"modelo":MODELO,"macro_f1":round(float(macro),4),"roc_auc":round(float(auc),4),"cm":cm.tolist()},
          open("resultados_bertimbau.json","w"), ensure_ascii=False, indent=2)

## 8. Gráficos: curva de loss e matriz de confusão

In [ ]:
import matplotlib.pyplot as plt
h=trainer.state.log_history
tr_l=[(x["epoch"],x["loss"]) for x in h if "loss" in x and "epoch" in x]
ev_l=[(x["epoch"],x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize=(6,4))
if tr_l: plt.plot(*zip(*tr_l), label="loss treino", color="#2f6fd0")
if ev_l: plt.plot(*zip(*ev_l), "o-", label="loss validação", color="#0f9d72")
plt.xlabel("Época"); plt.ylabel("Loss"); plt.title("Curva de loss — BERTimbau"); plt.legend(); plt.tight_layout()
plt.savefig("fig_loss_bertimbau.png", dpi=150); plt.show()

plt.figure(figsize=(4.4,3.8)); plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): plt.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black",fontsize=13)
plt.xticks([0,1],["esquerda","direita"]); plt.yticks([0,1],["esquerda","direita"])
plt.xlabel("Predito"); plt.ylabel("Real"); plt.title("Matriz de confusão — BERTimbau")
plt.colorbar(); plt.tight_layout(); plt.savefig("fig_matriz_confusao_bertimbau.png", dpi=150); plt.show()

## 9. Testar com texto novo

In [ ]:
def prever(texto):
    model.eval()
    e=tok(texto, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt").to(model.device)
    with torch.no_grad(): p=torch.softmax(model(**e).logits, dim=-1)[0]
    rot="DIREITA" if p[1]>p[0] else "ESQUERDA"
    return f"{rot}  (confiança {max(p).item()*100:.1f}%)"
for t in ["Defendo a redução de impostos e a liberdade econômica para o empreendedor brasileiro.",
          "É dever do Estado garantir saúde pública universal e reduzir a desigualdade social."]:
    print(prever(t), "<-", t[:60], "...")

## 10. Salvar o modelo e baixar resultados

In [ ]:
trainer.save_model("modelo_bertimbau_politica"); tok.save_pretrained("modelo_bertimbau_politica")
print("Modelo salvo em ./modelo_bertimbau_politica")
try:
    from google.colab import files
    for f in ["resultados_bertimbau.json","fig_loss_bertimbau.png","fig_matriz_confusao_bertimbau.png","discursos_camara.csv"]:
        files.download(f)
except Exception as e:
    print("Baixe pelos Arquivos à esquerda.", e)

## 11. Baixar a pasta do modelo para usar na pagina web (app.py)

O Colab nao baixa pastas direto, entao compactamos em .zip. Depois, no seu PC, descompacte dentro de `Trabalho_Robson` para ficar a pasta `modelo_bertimbau_politica` ao lado do `app.py`.

In [ ]:
import shutil
shutil.make_archive('modelo_bertimbau_politica', 'zip', 'modelo_bertimbau_politica')
print('zip criado: modelo_bertimbau_politica.zip')
try:
    from google.colab import files
    files.download('modelo_bertimbau_politica.zip')
except Exception as e:
    print('Baixe pelo painel de Arquivos a esquerda.', e)